# Hello, Numerai

Hello and welcome to the Numerai Data Science Tournament!

This notebook is designed to help you build your first machine learning model and start competing in the tournament.

In this notebook we will
1. Download and explore the Numerai dataset
2. Train your first machine learning model
3. Evaluate it with `CORR` and `BMC`
4. Deploy it to start making live submissions

In [ ]:
!python --version

In [ ]:
# Install dependencies
!pip install -q --upgrade numerapi pandas==2.3.1 pyarrow matplotlib lightgbm scipy cloudpickle==3.1.1

# Inline plots
%matplotlib inline

## 1. Dataset

The Numerai dataset is a tabular dataset that describes the stock market over time, compiled from high-quality (and expensive) data that would be difficult for individuals to obtain.

### Listing the datasets
Firstly, take a look at the files Numerai offers below:

In [ ]:
# Initialize NumerAPI - the official Python API client for Numerai
from numerapi import NumerAPI
napi = NumerAPI()

# List the datasets and available versions
all_datasets = napi.list_datasets()
dataset_versions = list(set(d.split('/')[0] for d in all_datasets))
print("Available versions:\n", dataset_versions)

# Set data version to the current dataset
DATA_VERSION = "v6.0"

# Print all files available for download for our version
current_version_files = [f for f in all_datasets if f.startswith(DATA_VERSION)]
print("Available", DATA_VERSION, "files:\n", current_version_files)

### Downloading datasets

The `features.json` file contains metadata about the features and targets available for training.

Let's download it and take a look:

In [ ]:
import json

# Download the feature metadata file
napi.download_dataset(f"{DATA_VERSION}/features.json")

# Read the metadata and display
feature_metadata = json.load(open(f"{DATA_VERSION}/features.json"))
for metadata in feature_metadata:
  print(metadata, len(feature_metadata[metadata]))

### Feature Sets & Groups
As you can see there are many features and targets to choose from.

Instead of training a model on all available features, let's pick a subset of features to analyze.

Here are a few starter sets Numerai offers:

- `small` contains a minimal subset of features

- `medium` contains a broader set of features

- `all` contains all available features

Let's compare their sizes:

In [ ]:
feature_sets = feature_metadata["feature_sets"]
for feature_set_name in ["small", "medium", "all"]:
  print(feature_set_name, len(feature_sets[feature_set_name]))

We will use the `small` feature set.

For this notebook, we will train on a sample of the training data. Using more rows takes longer and requires more memory.

In [ ]:
import pandas as pd

# Define our feature set. feature_country is text; LightGBM uses the numeric features here.
feature_set = [feature for feature in feature_sets["small"] if feature != "feature_country"]
assert feature_set, "The small feature set has no numeric features"

# Download the training data - this will take a few minutes
napi.download_dataset(f"{DATA_VERSION}/train.parquet")

# Load the features and the payout target for every training date
train = pd.read_parquet(
    f"{DATA_VERSION}/train.parquet",
    columns=["numerai_ticker", "date", "target"] + feature_set,
).dropna(subset=["target"])
assert not train.empty, "The training file has no mature target rows"
train = train.sample(n=min(200_000, len(train)), random_state=7)

### Training data

Each row represents a stock at a specific point in time:
- `numerai_ticker` identifies the stock
- `date` is the date
- `target` is a measure of future returns for that stock
- `features` describe the attributes of the stock for that date

In [ ]:
train

### Dates
Each `date` corresponds to a different week.

Think of the rows within a `date` as a single example - throughout this notebook we score everything "per date". The rows per date below trace the size of Numerai's stock universe over time.

In [ ]:
# Plot the number of rows per date
train.groupby("date").size().plot(
    title="Number of rows per date",
    figsize=(5, 3),
    xlabel="Date"
)

### Target
The `target` is a measure of stock-specific returns over the next 60 days. It is an alias for `target_jupiter_60` in this dataset.

Target values are binned into 5 unequal bins: `0`, `0.25`, `0.5`, `0.75`, `1.0`. This heavy regularization avoids overfitting because the underlying values are extremely noisy.

In [ ]:
# Plot density histogram of the target
train["target"].plot(
  kind="hist",
  title="Target",
  figsize=(5, 3),
  xlabel="Value",
  density=True,
  bins=50
)

### Features
The `features` are quantitative attributes of each stock: fundamentals like P/E ratio, technical signals like RSI, market data like short interest, analyst ratings, and much more. Numerai includes them because we believe they help predict the `target`.

When available, numeric feature values are binned into 5 equal-frequency bins: `0`, `1`, `2`, `3`, `4`. When a feature is missing for a date, that feature-date is filled with the middle bin `2`.

## 2. Modeling
Our task: predict the `target` using the `features`.

### Model training

You are free to use any tool or framework, but here we use LightGBM - a popular choice amongst tournament participants. While your model trains, watch this [video](https://www.youtube.com/watch?v=w8Y7hY05z7k) from our Chief Scientist MDO on why tree-based models work so well on tabular data.

In [ ]:
# https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html
import lightgbm as lgb

model = lgb.LGBMRegressor(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=5,
  num_leaves=2**5-1,
  colsample_bytree=0.1,
  verbosity=-1,
)

# This will take a few minutes per model 🍵
model.fit(
  train[feature_set],
  train["target"]
)

### Validation predictions

Now let's make out-of-sample predictions on the validation dataset.

To stay within Colab's memory limits we free the training data, filter out dates without mature targets, and predict one date at a time.

In [ ]:
# The target looks 60 days into the future, so embargo the first 12 weekly
# validation dates after training to avoid data leakage.
del train

napi.download_dataset(f"{DATA_VERSION}/validation.parquet")
validation_dates = sorted(pd.read_parquet(
    f"{DATA_VERSION}/validation.parquet", columns=["date"]
)["date"].unique())
scoring_dates = validation_dates[12:]
assert scoring_dates, "No validation dates remain after the embargo"

validation = pd.read_parquet(
    f"{DATA_VERSION}/validation.parquet",
    columns=["numerai_ticker", "date", "target"] + feature_set,
    filters=[("date", "in", scoring_dates)],
).dropna(subset=["target"])
assert not validation.empty, "No validation rows have mature targets"

# Generate predictions one date at a time
predictions = []
for date, date_data in validation.groupby("date"):
    predictions.append(pd.Series(model.predict(date_data[feature_set]), index=date_data.index))
validation["prediction"] = pd.concat(predictions)

# Keep only the columns we need for scoring
validation = validation[["numerai_ticker", "date", "prediction", "target"]]
validation

### Performance evaluation

Numerai's scores are designed to "align incentives": a model with good scores should help the hedge fund make good returns.

- `CORR` (`numerai_corr`): a Numerai-specific correlation between your predictions and the target.

- `BMC` (`correlation_contribution`): how uniquely additive your model is beyond Numerai's public benchmark models.

We'll start with `CORR`, then use `BMC` to compare with a public benchmark.

In [ ]:
# Install Numerai's open-source scoring tools
!pip install -q --no-deps numerai-tools

# Import the CORR scoring function
from numerai_tools.scoring import numerai_corr

### CORR: correlation with the target

`CORR` is the main "is this model directionally right?" metric - a proxy for portfolio returns if your predictions were used in live trading. Positive `CORR` means your model is aligned with Numerai's strategy.

Always score each `date` independently - "per date" metrics are what matter, not one score across the whole validation set.

And yes, the scores are low: +/- 0.05 is normal in quantitative finance. This is why we call Numerai the "hardest data science tournament on the planet".

In [ ]:
# Compute the per-date CORR between our predictions and target values
import matplotlib.pyplot as plt

per_date_corr = (
    validation.dropna(subset=["prediction", "target"])
    .groupby("date")[["prediction", "target"]]
    .apply(
        lambda date_data: numerai_corr(
            date_data[["prediction"]],
            date_data["target"]
        )
    )["prediction"]
)

# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

per_date_corr.plot(title="Validation CORR", kind="bar", ax=axes[0], xticks=[], legend=False, snap=False)
per_date_corr.cumsum().plot(title="Cumulative Validation CORR", kind="line", ax=axes[1], legend=False)
plt.tight_layout()
plt.show()

Cumulative `CORR` is easier to read than the raw per-date scores - think of it as a rough backtest of your model's target alignment over the validation period.

### CORR summary metrics

A few summary metrics over the entire validation period:

- `Mean`: long-term performance.

- `Sharpe`: consistency - mean divided by standard deviation.

- `Max drawdown`: risk - the largest peak-to-trough drop in cumulative score.

In [ ]:
# Compute CORR performance metrics
corr_mean = per_date_corr.mean()
corr_std = per_date_corr.std(ddof=0)
corr_sharpe = corr_mean / corr_std
corr_max_drawdown = (
    per_date_corr.cumsum().expanding(min_periods=1).max()
    - per_date_corr.cumsum()
).max()

pd.DataFrame({
    "mean": [corr_mean],
    "std": [corr_std],
    "sharpe": [corr_sharpe],
    "max_drawdown": [corr_max_drawdown]
}, index=["CORR"]).T

### MMC: Meta Model Contribution

`MMC` measures whether your model adds signal after neutralizing away everything it shares with the Meta Model.

Careful, though: the Meta Model is a moving target - it changes as the crowd adopts new data and modeling ideas.

### BMC: Benchmark Model Contribution

`BMC` uses the same `correlation_contribution` calculation as `MMC`, but replaces the ever-changing Meta Model with a fixed public benchmark. We use `v6_lgbm_jupiter60` here.

This makes `BMC` a practical research objective: does your model add signal beyond a strong, modern benchmark that anyone can download?

In [ ]:
# Import the contribution scoring function
from numerai_tools.scoring import correlation_contribution

# Download and join the public v6 benchmark predictions
BENCHMARK_MODEL = "v6_lgbm_jupiter60"
napi.download_dataset(f"{DATA_VERSION}/validation_benchmark_models.parquet")
benchmarks = pd.read_parquet(
    f"{DATA_VERSION}/validation_benchmark_models.parquet",
    columns=["numerai_ticker", "date", BENCHMARK_MODEL],
)
validation = validation.merge(benchmarks, on=["numerai_ticker", "date"], how="left", validate="one_to_one")

# Calculate BMC for dates covered by the benchmark
per_date_bmc = (
    validation.dropna(subset=["prediction", "target", BENCHMARK_MODEL])
    .groupby("date")[["prediction", BENCHMARK_MODEL, "target"]]
    .apply(
        lambda date_data: correlation_contribution(
            date_data[["prediction"]],
            date_data[BENCHMARK_MODEL],
            date_data["target"]
        )
    )["prediction"]
)

### BMC validation performance

We can use `BMC` as an offline report card for contribution. The fixed benchmark is published for the validation window:

In [ ]:
# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

per_date_bmc.plot(title="Validation BMC", kind="bar", ax=axes[0], xticks=[], legend=False, snap=False)
per_date_bmc.cumsum().plot(title="Cumulative Validation BMC", kind="line", ax=axes[1], legend=False)
plt.tight_layout()
plt.show()

### Current contribution summary metrics

The table below summarizes BMC over the validation dates.

In [ ]:
bmc_summary = pd.DataFrame({
    "mean": [per_date_bmc.mean()],
    "std": [per_date_bmc.std(ddof=0)],
    "sharpe": [per_date_bmc.mean() / per_date_bmc.std(ddof=0)],
    "max_drawdown": [(per_date_bmc.cumsum().cummax() - per_date_bmc.cumsum()).max()],
}, index=["BMC"]).T
bmc_summary

These metrics don't need to be amazing - we're just getting started. The next tutorials cover improving target prediction while adding unique signal beyond public benchmarks.

## 3. Submissions

Unlike Kaggle competitions that evaluate models on <ins>test</ins> performance, Numerai evaluates models on <ins>live</ins> performance.

### Live predictions

New `live features` are released regularly - the current state of the stock market. Your job is to predict the target: stock-specific returns 60 days into the future.

In [ ]:
# Download latest live features
napi.download_dataset(f"{DATA_VERSION}/live.parquet")

# Load live features
live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_set)

# Generate live predictions
live_predictions = model.predict(live_features[feature_set])

# Format submission
pd.Series(live_predictions, index=live_features.index).rank(pct=True).to_frame("prediction")

### Model upload

To automate this process, you can simply:
- Define your prediction pipeline as a function
- Serialize your function using the `cloudpickle` library
- Upload your model pickle file to Numerai
- Let Numerai run your model to submit live predictions every day

Read more about Model Uploads and other self-hosted automation options in our [docs](https://docs.numer.ai/numerai-tournament/submissions#automation).

In [ ]:
# Define your prediction pipeline as a function
def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    live_predictions = model.predict(live_features[feature_set])
    submission = pd.Series(live_predictions, index=live_features.index).rank(pct=True)
    return submission.to_frame("prediction")

In [ ]:
# Use the cloudpickle library to serialize your function
import cloudpickle
p = cloudpickle.dumps(predict)
with open("tutorial_basics.pkl", "wb") as f:
    f.write(p)

In [ ]:
# Download file if running in Google Colab
try:
    from google.colab import files
    files.download('tutorial_basics.pkl')
except ImportError:
    print('Model saved as tutorial_basics.pkl')

That's it! You now have a pickle file that is ready for upload.

Head back to the [Tournament Basics onboarding step](https://numer.ai/onboarding/tournament-basics) to upload your model!